In [ ]:
import ctypes, os, numpy as np, open3d as o3d
from time import perf_counter

densify = ctypes.CDLL(os.path.abspath("densify.dll")).densify_point_flags
densify.restype  = None
densify.argtypes = [ctypes.POINTER(ctypes.c_float),
                    ctypes.c_int, ctypes.c_int,
                    ctypes.c_float, ctypes.c_float,
                    ctypes.POINTER(ctypes.c_ubyte)]

rng = np.random.default_rng(42)

line_x = np.linspace(-800, 800, 5000, dtype=np.float32)
line = np.stack([line_x, rng.normal(0,  3, size=5000), rng.normal(0,  3, size=5000)], axis=1)

plane_y = rng.uniform(-400, 400, 10_000).astype(np.float32)
plane_z = rng.uniform(-400, 400, 10_000).astype(np.float32)
plane = np.stack([np.full_like(plane_y, 600, dtype=np.float32), plane_y, plane_z], axis=1)

phi = rng.uniform(0, 2*np.pi, 20_000)
costh = rng.uniform(-1, 1,       20_000)
sinth = np.sqrt(1-costh**2)
r = rng.normal(300, 40,      20_000)          
sphere = np.stack([r*sinth*np.cos(phi), r*sinth*np.sin(phi), r*costh], axis=1).astype(np.float32)

pts_np = np.vstack([line, plane, sphere]).astype(np.float32)
n = pts_np.shape[0]
print(f"Composite cloud  →  {n:,} points")

tiles, th1, th2 = 64, 50.0, 0.5    

pts_for_cuda = pts_np.T.ravel(order="C")  
flags        = np.empty(n, dtype=np.uint8)


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Composite cloud  →  35,000 points


In [ ]:
t0 = perf_counter()
densify(pts_for_cuda.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
        ctypes.c_int(n), ctypes.c_int(tiles),
        ctypes.c_float(th1), ctypes.c_float(th2),
        flags.ctypes.data_as(ctypes.POINTER(ctypes.c_ubyte)))
t1 = perf_counter()

print(f"CUDA finished in {(t1-t0)*1e3:.2f} ms   "
      f"(true = {flags.sum():,} , false = {n-flags.sum():,})")

col = np.zeros((n,3), float)
col[flags==1] = [0,0,1]   # blue  → densify
col[flags==0] = [1,0,0]   # red   → normal

pcd = o3d.geometry.PointCloud()
pcd.points  = o3d.utility.Vector3dVector(pts_np.astype(np.float64))
pcd.colors  = o3d.utility.Vector3dVector(col)
o3d.visualization.draw_geometries([pcd],
        window_name="blue = densify, red = normal",
        width=1000, height=700)

CUDA finished in 1436.35 ms   (true = 9,339 , false = 25,661)


In [ ]:
import ctypes, os, sys, numpy as np, open3d as o3d
from time import perf_counter

DLL_PATH = os.path.abspath("densify.dll")
densify  = ctypes.CDLL(DLL_PATH).densify_point_flags
densify.restype  = None
densify.argtypes = [ctypes.POINTER(ctypes.c_float),
                    ctypes.c_int, ctypes.c_int,
                    ctypes.c_float, ctypes.c_float,
                    ctypes.POINTER(ctypes.c_ubyte)]

tiles = 64          # grid resolution per axis
th1   = 0        # variance   threshold
th2   = 0.475        # L-P-S score threshold

def load_mesh():
    if len(sys.argv) > 1 and os.path.isfile(sys.argv[1]):
        return o3d.io.read_triangle_mesh(sys.argv[1])

    try:
        import open3d.data as o3d_data
        bunny_data = o3d_data.BunnyMesh()
        return o3d.io.read_triangle_mesh(bunny_data.path)
    except Exception as e:
        print("[warn] Open3D sample download failed:", e)

    fallback = "bun_zipper.ply"
    if os.path.isfile(fallback):
        return o3d.io.read_triangle_mesh(fallback)

    raise RuntimeError(
        "Could not find a bunny mesh.\n"
        "• give a path:  python bunny_densify.py path/to/mesh.ply\n"
        "• or place 'bun_zipper.ply' next to the script\n"
        "• or ensure Open3D can download its sample dataset")

mesh = load_mesh()
mesh.compute_vertex_normals()

N_POINTS = 120_000
pcd = mesh.sample_points_poisson_disk(number_of_points=N_POINTS)
pts_np = np.asarray(pcd.points, dtype=np.float32)      # (N,3)
n = pts_np.shape[0]
print(f"Loaded mesh → sampled point cloud with {n:,} points")

pts_for_cuda = pts_np.T.astype(np.float32).ravel(order="C")
flags = np.empty(n,  dtype=np.uint8)




Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loaded mesh → sampled point cloud with 120,000 points


In [ ]:
t0 = perf_counter()
densify(pts_for_cuda.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
        ctypes.c_int(n), ctypes.c_int(tiles),
        ctypes.c_float(th1), ctypes.c_float(th2),
        flags.ctypes.data_as(ctypes.POINTER(ctypes.c_ubyte)))
t1 = perf_counter()

true_cnt  = int(flags.sum())
false_cnt = n - true_cnt
print(f"CUDA done in {(t1-t0)*1e3:.2f} ms"
      f"   (True={true_cnt:,} | False={false_cnt:,})")

col = np.zeros((n,3), dtype=np.float64)
col[flags == 1] = [0.0, 0.0, 1.0]      # densify → blue
col[flags == 0] = [1.0, 0.0, 0.0]      # normal  → red
pcd.colors = o3d.utility.Vector3dVector(col)

print("Showing original (grey) and coloured result…\n"
      "Close first window to see the second.")
orig_pcd = o3d.geometry.PointCloud()
orig_pcd.points = mesh.vertices                      # copy vertices
orig_pcd.colors = o3d.utility.Vector3dVector(
                        np.full((len(mesh.vertices),3), 0.6))  # light-grey
o3d.visualization.draw_geometries([orig_pcd], window_name="Original cloud (grey)")


o3d.visualization.draw_geometries([pcd], window_name="blue = densify  |  red = normal")

CUDA done in 10.62 ms   (True=73,587 | False=46,413)
Showing original (grey) and coloured result…
Close first window to see the second.
